In [ ]:
def compute_metrics(eval_pred):
    """transformers Trainer에서 사용할 metrics 함수 (OverflowError 수정)"""
    predictions, labels = eval_pred
    
    # 토크나이저가 없으면 기본 메트릭 반환
    if TOKENIZER_GLOBAL is None:
        return {}
    
    try:
        # predictions가 logits인 경우 argmax 처리
        if len(predictions.shape) == 3:
            predictions = np.argmax(predictions, axis=-1)
        
        # 유효한 토큰 ID로 필터링 (음수나 vocab_size를 초과하는 ID 제거)
        vocab_size = len(TOKENIZER_GLOBAL.get_vocab())
        
        # predictions 처리
        valid_predictions = []
        for pred_seq in predictions:
            # 유효한 범위의 토큰만 유지
            valid_tokens = [token_id for token_id in pred_seq if 0 <= token_id < vocab_size]
            valid_predictions.append(valid_tokens)
        
        # labels 처리 (-100을 pad_token_id로 변경)
        valid_labels = []
        for label_seq in labels:
            # -100을 pad_token_id로 변경하고 유효한 토큰만 유지
            processed_tokens = []
            for token_id in label_seq:
                if token_id == -100:
                    continue  # -100은 무시
                elif 0 <= token_id < vocab_size:
                    processed_tokens.append(token_id)
            valid_labels.append(processed_tokens)
        
        # 빈 시퀀스 체크
        if not valid_predictions or not valid_labels:
            return {
                'rouge1_fmeasure': 0.0,
                'rouge2_fmeasure': 0.0,
                'rougeL_fmeasure': 0.0,
                'rouge_avg': 0.0,
            }
        
        # 토큰 ID를 텍스트로 변환 (안전한 방식)
        decoded_preds = []
        for pred_tokens in valid_predictions:
            if pred_tokens:  # 빈 시퀀스가 아닌 경우만
                try:
                    decoded_text = TOKENIZER_GLOBAL.decode(pred_tokens, skip_special_tokens=True)
                    decoded_preds.append(decoded_text)
                except Exception as e:
                    print(f"⚠️ Prediction decode error: {e}")
                    decoded_preds.append("")  # 빈 문자열로 대체
            else:
                decoded_preds.append("")
        
        decoded_labels = []
        for label_tokens in valid_labels:
            if label_tokens:  # 빈 시퀀스가 아닌 경우만
                try:
                    decoded_text = TOKENIZER_GLOBAL.decode(label_tokens, skip_special_tokens=True)
                    decoded_labels.append(decoded_text)
                except Exception as e:
                    print(f"⚠️ Label decode error: {e}")
                    decoded_labels.append("")  # 빈 문자열로 대체
            else:
                decoded_labels.append("")
        
        # 빈 문자열 필터링
        filtered_preds = []
        filtered_labels = []
        for pred, label in zip(decoded_preds, decoded_labels):
            if pred.strip() and label.strip():  # 둘 다 비어있지 않은 경우만
                filtered_preds.append(pred.strip())
                filtered_labels.append(label.strip())
        
        if not filtered_preds or not filtered_labels:
            return {
                'rouge1_fmeasure': 0.0,
                'rouge2_fmeasure': 0.0,
                'rougeL_fmeasure': 0.0,
                'rouge_avg': 0.0,
            }
        
        # ROUGE 점수 계산 (안전한 방식)
        rouge_scores = calculate_rouge_scores(filtered_preds, filtered_labels)
        
        # 결과 정리
        result = {
            'rouge1_fmeasure': rouge_scores['rouge-1']['f'] * 100,
            'rouge2_fmeasure': rouge_scores['rouge-2']['f'] * 100,
            'rougeL_fmeasure': rouge_scores['rouge-l']['f'] * 100,
        }
        
        # 평균 ROUGE 점수
        avg_rouge = (result['rouge1_fmeasure'] + result['rouge2_fmeasure'] + result['rougeL_fmeasure']) / 3
        result['rouge_avg'] = avg_rouge
        
        return result
        
    except Exception as e:
        print(f"⚠️ Metrics 계산 오류: {e}")
        # 오류 발생시 기본값 반환
        return {
            'rouge1_fmeasure': 0.0,
            'rouge2_fmeasure': 0.0,
            'rougeL_fmeasure': 0.0,
            'rouge_avg': 0.0,
        }